# SCA 2.0 — Adapter Usage Demo

**EconLLM Lab** · [`SCA2_PofW`](https://github.com/EconLLM-Lab/SCA2_PofW)

This notebook shows how to **load and use the country-specific DPO+QLoRA adapters** produced by the SCA 2.0 pipeline (`DPO_train_test/`) for three supported modes:

1. **Option-likelihood scoring** (the canonical WVS evaluation protocol — see `DPO_eval_WVS/`): score every valid response code and normalize to a response distribution.
2. **DPO implied reward recovery** (Tier-1 diagnostics): recover the implied utility margin $\Delta r = \beta\,[\log\frac{\pi_\theta(y_w)}{\pi_{ref}(y_w)} - \log\frac{\pi_\theta(y_l)}{\pi_{ref}(y_l)}]$ on preference pairs.
3. **Free-form generation** (exploratory only — note that option-likelihood validation does *not* certify open-ended conversational authenticity).

## Before you run

- Set `BASE_MODEL_NAME` (default `meta-llama/Llama-3.1-8B-Instruct`) and authenticate with Hugging Face (`notebook_login()`).
- Point `ADAPTER_DIRS` at the trained adapters (default layout: `/content/drive/MyDrive/DPO/dpo_qlora_adapter_llama3_3594_{US|MEX}_3` — the same naming used by `DPO_train.ipynb`).
- Runtime: a single Colab-class GPU (T4 or better) suffices; 4-bit NF4 quantization keeps memory ~ 8 GB.

## 1. Install and imports

In [ ]:
!pip -q install -U "transformers>=4.41.0" "accelerate>=0.30.0" "peft>=0.11.1" "bitsandbytes>=0.46.1" "scipy>=1.10.0"

In [ ]:
import math
from pathlib import Path

import numpy as np
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

# Optional: Colab Drive mount if adapters live on Drive.
# from google.colab import drive
# drive.mount("/content/drive")

BASE_MODEL_NAME = "meta-llama/Llama-3.1-8B-Instruct"
ADAPTER_DIRS = {
    "US": "/content/drive/MyDrive/DPO/dpo_qlora_adapter_llama3_3594_US_3",
    "MEX": "/content/drive/MyDrive/DPO/dpo_qlora_adapter_llama3_3594_MEX_3",
}

BETA = 0.1   # DPO temperature used in training; keep consistent with DPO_train.ipynb

## 2. Load the base model + adapter (PEFT)

In [ ]:
def load_adapter_model(adapter_dir):
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto",
    )
    adapter_model = PeftModel.from_pretrained(base_model, adapter_dir)
    adapter_model.eval()
    return adapter_model, tokenizer


adapter_model, tokenizer = load_adapter_model(ADAPTER_DIRS["MEX"])
print("Loaded MEX adapter:", adapter_model.active_adapter)
print("Trainable params:", sum(p.numel() for p in adapter_model.parameters() if p.requires_grad))

## 3. Mode A — Option-likelihood scoring

This is the canonical evaluation protocol used in `DPO_eval_WVS/`: for a survey question with $K$ response codes, compute the completion log-probability of each code and softmax-normalize into a response distribution $\hat P = (\hat p_1, \dots, \hat p_K)$. Country names are **not** mentioned in the prompt, so any country-specific behavior must come from the adapter itself.

In [ ]:
def sequence_logprob(model, tokenizer, prompt_text, completion_text):
    """Sum log p(completion | prompt) over completion tokens only."""
    model.eval()
    device = next(model.parameters()).device

    formatted_prompt = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt_text}],
        tokenize=False,
        add_generation_prompt=True,
    )
    prompt_ids = tokenizer(formatted_prompt, add_special_tokens=False).input_ids
    completion_ids = tokenizer(completion_text, add_special_tokens=False).input_ids
    if not completion_ids:
        raise ValueError("Empty completion")

    input_ids = torch.tensor([prompt_ids + completion_ids], device=device)
    labels = torch.tensor([[-100] * len(prompt_ids) + completion_ids], device=device)

    with torch.no_grad():
        logits = model(input_ids=input_ids).logits
    shifted_logits, shifted_labels = logits[:, :-1, :], labels[:, 1:]
    log_probs = torch.log_softmax(shifted_logits, dim=-1)
    mask = shifted_labels.ne(-100)
    safe_labels = shifted_labels.masked_fill(~mask, 0)
    token_log_probs = log_probs.gather(-1, safe_labels.unsqueeze(-1)).squeeze(-1)
    return float((token_log_probs * mask).sum().detach().cpu())


def score_survey_question(model, tokenizer, question_text, codes, labels):
    prompt = (
        "Answer this questionnaire as an individual person. "
        "Respond naturally and sincerely, as someone would in real life. "
        "Do not mention being an AI or assistant.\n\n"
        f"Question:\n{question_text}\n\n"
        "Select exactly one response code from the options below.\n"
        + "\n".join(f"{c}: {l}" for c, l in zip(codes, labels))
        + "\n\nReturn only the response code, with no words or explanation.\n\nAnswer:"
    )
    logprobs = [
        sequence_logprob(model, tokenizer, prompt, f" {code}")
        for code in codes
    ]
    probs = np.exp(np.array(logprobs) - np.max(logprobs))
    probs = probs / probs.sum()
    return probs, prompt


# Example: generalized trust question (WVS Wave 7 Q57).
q_text = "Generally speaking, would you say that most people can be trusted or that you need to be very careful in dealing with people?"
codes = ["1", "2"]
labels = ["Most people can be trusted", "Need to be very careful"]

probs, _ = score_survey_question(adapter_model, tokenizer, q_text, codes, labels)
for c, p in zip(codes, probs):
    print(f"code {c}: {p:.3f}")
print("\nWVS 2017-2022 Mexico benchmark (approx.): trust ~15%, careful ~85%")

## 4. Mode B — DPO implied reward recovery

Tier-1 diagnostic: for a preference pair $(q, y_w, y_l)$, compute the implied reward margin using the **reference model (adapter disabled)** as baseline. Positive $\Delta r$ means the trained policy ranks the culture-conditioned chosen response above the rejected one.

In [ ]:
def dpo_implied_reward_delta(model, tokenizer, prompt, chosen, rejected, beta=BETA):
    with model.disable_adapter():
        ref_chosen = sequence_logprob(model, tokenizer, prompt, chosen)
        ref_rejected = sequence_logprob(model, tokenizer, prompt, rejected)
    adapter_chosen = sequence_logprob(model, tokenizer, prompt, chosen)
    adapter_rejected = sequence_logprob(model, tokenizer, prompt, rejected)

    ref_margin = ref_chosen - ref_rejected
    adapter_margin = adapter_chosen - adapter_rejected
    reward_delta = beta * (adapter_margin - ref_margin)
    pref_prob = 1.0 / (1.0 + math.exp(-reward_delta))
    return {
        "ref_margin": ref_margin,
        "adapter_margin": adapter_margin,
        "dpo_reward_delta": reward_delta,
        "dpo_pref_prob": pref_prob,
        "dpo_prefers_chosen": reward_delta > 0,
    }


# A minimal pair; in the real pipeline these come from the synthetic triplet files.
prompt = "You are considering how much to trust a stranger with a favor. What do you do?"
chosen = "I would be cautious and not rely on them."
rejected = "I would trust them fully and without hesitation."
out = dpo_implied_reward_delta(adapter_model, tokenizer, prompt, chosen, rejected)
print(out)

## 5. Mode C — Free-form generation (exploratory)

> **Caveat:** option-likelihood validation does *not* certify open-ended conversational authenticity. Treat free-form outputs as exploratory only.

In [ ]:
def generate_answer(model, tokenizer, prompt_text, max_new_tokens=80):
    messages = [{"role": "user", "content": prompt_text}]
    encoded = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    )
    encoded = encoded.to(next(model.parameters()).device)
    terminators = [
        t for t in [tokenizer.eos_token_id, tokenizer.convert_tokens_to_ids("<|eot_id|>")]
        if t is not None and t != tokenizer.unk_token_id
    ]
    with torch.no_grad():
        out = model.generate(
            encoded,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=terminators,
            use_cache=True,
        )
    return tokenizer.decode(out[0][encoded.shape[-1]:], skip_special_tokens=True)


print(generate_answer(adapter_model, tokenizer, "How much do you trust people you meet for the first time? Answer briefly."))